In [ ]:
import json
import os
import platform
import subprocess
import sys
import tempfile
from pathlib import Path

out = Path("/kaggle/working")
workspace = Path(tempfile.mkdtemp(prefix="clean-vllm-"))
(workspace / "src/runtime").mkdir(parents=True)
(workspace / "scripts").mkdir()
os.chdir(workspace)
gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,uuid,driver_version,compute_cap", "--format=csv,noheader"], text=True).strip()
print(gpu_info)
(out / "environment.json").write_text(json.dumps({"gpu": gpu_info, "python": sys.version, "platform": platform.platform()}, indent=2))
capability = gpu_info.splitlines()[0].rsplit(",", 1)[-1].strip()
assert tuple(map(int, capability.split("."))) >= (7, 5), "vLLM 0.26.0 requires CC 7.5 or higher"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
%%writefile src/runtime/sliding_window_probe.py
"""Observe vLLM's native per-request block release at the scheduler boundary."""


class SlidingWindowEvidenceError(RuntimeError):
    """The observed scheduler trace does not establish the requested condition."""


def observe_block_release(scheduler, trace):
    """Wrap the native coordinator call without changing any scheduler or cache decisions."""
    cache = scheduler.kv_cache_manager
    coordinator = cache.coordinator
    pool = cache.block_pool
    native_remove = coordinator.remove_skipped_blocks

    def observe(request_id, processed_computed_tokens, num_prompt_tokens=None):
        request = scheduler.requests[request_id]
        (blocks,) = coordinator.get_blocks(request_id)
        previous = tuple(blocks)
        before = [None if block.is_null else block.block_id for block in previous]
        refs_before = [block.ref_cnt for block in previous]
        free_before = pool.get_num_free_blocks()
        running = request.status.name == "RUNNING"
        output_tokens = request.num_output_tokens

        result = native_remove(request_id, processed_computed_tokens, num_prompt_tokens)

        (blocks,) = coordinator.get_blocks(request_id)
        after = [None if block.is_null else block.block_id for block in blocks]
        free_after = pool.get_num_free_blocks()
        released = []
        if before != after:
            free_ids = {block.block_id for block in pool.free_block_queue.get_all_free_blocks()}
            for position, (old, new) in enumerate(zip(before, after)):
                if old is not None and new is None:
                    released.append(dict(
                        position=position, block_id=old,
                        ref_before=refs_before[position], ref_after=previous[position].ref_cnt,
                        in_free_queue=old in free_ids,
                    ))

        trace.append(dict(
            request_id=request_id, processed_tokens=processed_computed_tokens,
            output_tokens=output_tokens, running=running,
            preemptions=request.num_preemptions, before=before, after=after,
            free_before=free_before, free_after=free_after, released=released,
        ))
        return result

    coordinator.remove_skipped_blocks = observe
    return native_remove


def validate_trace(condition, trace, *, block_size, window, prompt_tokens, output_tokens):
    """Require a live release and further decode, or a matched retaining control."""
    if not trace or len({row["request_id"] for row in trace}) != 1:
        raise SlidingWindowEvidenceError("Expected scheduler observations for one request")
    if any(row["preemptions"] for row in trace):
        raise SlidingWindowEvidenceError("The request was preempted")
    decode = [row for row in trace if row["running"] and row["processed_tokens"] >= prompt_tokens
              and row["output_tokens"] > 0]
    if not decode or decode[0]["processed_tokens"] != prompt_tokens or decode[0]["output_tokens"] != 1:
        raise SlidingWindowEvidenceError("Missing first decode step after ordinary full prefill")
    first = decode[0]
    expected_prefill_blocks = (prompt_tokens + block_size - 1) // block_size
    if (len(first["before"]) != expected_prefill_blocks or None in first["before"]
            or first["before"] != first["after"]):
        raise SlidingWindowEvidenceError("Missing initial full-cache ownership")
    if (decode[-1]["processed_tokens"] < window - 1 + block_size
            or decode[-1]["output_tokens"] != output_tokens - 1):
        raise SlidingWindowEvidenceError("Insufficient observed decode across the release boundary")

    releases = []
    for row in trace:
        before, after = row["before"], row["after"]
        removed = {index for index, block in enumerate(before) if block is not None
                   and index < len(after) and after[index] is None}
        if len(before) != len(after) or any(old != new and index not in removed
                                           for index, (old, new) in enumerate(zip(before, after))):
            raise SlidingWindowEvidenceError("Unexpected block-table mutation")
        if not removed:
            if row["free_before"] != row["free_after"] or row["released"]:
                raise SlidingWindowEvidenceError("Unexplained movement in the free queue")
            continue
        if condition != "sliding" or row not in decode:
            raise SlidingWindowEvidenceError("Unexpected release outside live sliding-window decode")
        if row["free_after"] - row["free_before"] != len(removed):
            raise SlidingWindowEvidenceError("Free-block count did not increase as expected")
        expired_before = row["processed_tokens"] - window + 1
        verified = {entry["position"] for entry in row["released"]
                    if entry["position"] in removed
                    and before[entry["position"]] == entry["block_id"]
                    and entry["ref_before"] == 1 and entry["ref_after"] == 0
                    and entry["in_free_queue"]
                    and (entry["position"] + 1) * block_size <= expired_before}
        if verified != removed or len(row["released"]) != len(removed):
            raise SlidingWindowEvidenceError("Missing out-of-window ownership-to-free-queue transition")
        releases.append(row)

    if condition == "sliding":
        if not releases:
            raise SlidingWindowEvidenceError("Native block release was not observed")
        if not any(row["output_tokens"] > releases[0]["output_tokens"] for row in decode):
            raise SlidingWindowEvidenceError("No later decode after the first native release")
    elif len(decode[-1]["after"]) <= expected_prefill_blocks or None in decode[-1]["after"]:
        raise SlidingWindowEvidenceError("Full-attention control did not retain its growing cache")
    return len(releases)


In [ ]:
%%writefile scripts/run_vllm_sliding_window_probe.py
"""Run one clean native sliding-window or full-attention vLLM diagnostic."""

import argparse
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import sys
import traceback

from src.runtime.sliding_window_probe import (
    SlidingWindowEvidenceError,
    observe_block_release,
    validate_trace,
)


VERSION = "0.26.0"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
PROMPT = "The first seven positive integers are 1, 2, 3, 4,"
WINDOW = 32
MAX_TOKENS = 80
SNAPSHOT_HASHES = {
    "model.safetensors": "fdf756fa7fcbe7404d5c60e26bff1a0c8b8aa1f72ced49e7dd0210fe288fb7fe",
    "config.json": "18e18afcaccafade98daf13a54092927904649e1dd4eba8299ab717d5d94ff45",
    "tokenizer.json": "c0382117ea329cdf097041132f6d735924b697924d6f6fc3945713e96ce87539",
}


def inspect_attention_layers(model):
    from vllm.model_executor.layers.attention import Attention

    return {
        layer.layer_name: dict(
            backend=layer.get_attn_backend().get_name(), window=layer.sliding_window,
            dtype=str(layer.kv_cache.dtype), device=layer.kv_cache.device.type,
            shape=list(layer.kv_cache.shape), elements=layer.kv_cache.numel(),
        )
        for layer in model.modules() if isinstance(layer, Attention)
    }


def run(args, record):
    if platform.system() != "Linux":
        raise SlidingWindowEvidenceError("This vLLM probe requires Linux")
    version = importlib.metadata.version("vllm")
    variants = ("cpu",) if args.device == "cpu" else ("cu128", "cu129", "cu130")
    versions = {VERSION} | {f"{VERSION}+{variant}" for variant in variants}
    if version not in versions:
        raise SlidingWindowEvidenceError(f"Expected one of {sorted(versions)}, found {version}")

    model_dir = args.model_dir.resolve(strict=True)
    for name, expected in SNAPSHOT_HASHES.items():
        digest = hashlib.sha256()
        with (model_dir / name).open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024**2), b""):
                digest.update(chunk)
        if digest.hexdigest() != expected:
            raise SlidingWindowEvidenceError(f"Clean snapshot fingerprint mismatch for {name}")
    # The observer needs direct access to the single in-process scheduler.
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ["HF_HUB_OFFLINE"] = "1"

    import torch
    from vllm import LLM, SamplingParams
    from vllm.platforms import current_platform
    from vllm.v1.core.single_type_kv_cache_manager import FullAttentionManager, SlidingWindowManager
    from vllm.v1.kv_cache_interface import FullAttentionSpec, SlidingWindowSpec

    if current_platform.device_type != args.device:
        raise SlidingWindowEvidenceError(f"Resolved vLLM platform is {current_platform.device_type}")
    if args.device == "cuda" and not torch.cuda.is_available():
        raise SlidingWindowEvidenceError("CUDA is unavailable")

    cpu = args.device == "cpu"
    dtype = "float32" if cpu else "float16"
    block_size = 32 if cpu else 16
    backend = "CPU_ATTN" if cpu else "TRITON_ATTN"
    sliding = args.condition == "sliding"
    window = WINDOW if sliding else None
    overrides = {"sliding_window": window, "use_sliding_window": sliding}
    record.update(
        device=args.device, hardware=platform.processor() if cpu else torch.cuda.get_device_name(),
        torch_version=torch.__version__,
        vllm_version=version,
        transformers_version=importlib.metadata.version("transformers"),
        model=MODEL, model_revision=REVISION, verified_snapshot_hashes=SNAPSHOT_HASHES,
        model_dtype=dtype, cache_dtype_requested="auto", hf_overrides=overrides,
        attention_backend=backend, seed=42, max_output_tokens=MAX_TOKENS,
    )
    if not cpu:
        record["cuda"] = dict(torch_build=torch.version.cuda, capability=torch.cuda.get_device_capability())
    llm = LLM(
        model=str(model_dir), tokenizer=str(model_dir),
        trust_remote_code=False, enable_lora=False, dtype=dtype, hf_overrides=overrides,
        kv_cache_dtype="auto", kv_cache_memory_bytes=32 * 1024**2,
        enforce_eager=True, async_scheduling=False, enable_prefix_caching=False,
        disable_hybrid_kv_cache_manager=False, max_model_len=128,
        max_num_batched_tokens=128, max_num_seqs=1, block_size=block_size,
        attention_backend=backend,
        generation_config="vllm", seed=42,
    )
    core = llm.llm_engine.engine_core.engine_core
    scheduler = core.scheduler
    (manager,) = scheduler.kv_cache_manager.coordinator.single_type_managers
    (group,) = scheduler.kv_cache_config.kv_cache_groups
    spec = manager.kv_cache_spec
    record["resolved_cache"] = dict(
        manager=type(manager).__name__, spec=type(spec).__name__, dtype=str(spec.dtype),
        sliding_window=getattr(spec, "sliding_window", None), block_size=manager.block_size,
        layers=len(group.layer_names), page_bytes_per_layer=spec.page_size_bytes,
        pool_total_blocks=scheduler.kv_cache_manager.block_pool.num_gpu_blocks,
    )
    expected_manager = SlidingWindowManager if sliding else FullAttentionManager
    expected_spec = SlidingWindowSpec if sliding else FullAttentionSpec
    if (type(manager) is not expected_manager or type(spec) is not expected_spec
            or getattr(spec, "sliding_window", None) != window or spec.dtype != getattr(torch, dtype)
            or len(group.layer_names) != 24):
        raise SlidingWindowEvidenceError("The scheduler did not resolve the required cache policy")

    prompt_ids = llm.get_tokenizer().encode(PROMPT, add_special_tokens=False)
    if not 0 < len(prompt_ids) < WINDOW:
        raise SlidingWindowEvidenceError("Prompt must fit inside the native sliding window")
    record["prompt_token_ids"] = prompt_ids
    trace = record["trace"]
    restore = observe_block_release(scheduler, trace)
    try:
        (response,) = llm.generate(
            {"prompt_token_ids": prompt_ids},
            SamplingParams(temperature=0, max_tokens=MAX_TOKENS, ignore_eos=True),
            use_tqdm=False,
        )
    finally:
        scheduler.kv_cache_manager.coordinator.remove_skipped_blocks = restore
    (output,) = response.outputs
    record["generated_token_ids"] = list(output.token_ids)
    record["finish_reason"] = output.finish_reason
    if (not response.finished or output.finish_reason != "length" or len(output.token_ids) != MAX_TOKENS
            or response.prompt_token_ids != prompt_ids):
        raise SlidingWindowEvidenceError("The engine did not complete the fixed diagnostic request")
    (layers,) = llm.apply_model(inspect_attention_layers)
    record["observed_attention_layers"] = layers
    if set(layers) != set(group.layer_names) or any(
        layer["backend"] != backend or layer["window"] != window
        or layer["dtype"] != str(spec.dtype) or layer["device"] != args.device or layer["elements"] <= 0
        for layer in layers.values()
    ):
        raise SlidingWindowEvidenceError("Attention layers did not attest the expected backend and KV tensors")
    record["verified_release_steps"] = validate_trace(
        args.condition, trace, block_size=manager.block_size, window=WINDOW,
        prompt_tokens=len(prompt_ids), output_tokens=len(output.token_ids),
    )
    last = next(row for row in reversed(trace) if row["running"] and row["output_tokens"] > 0)
    owned = sum(block_id is not None for block_id in last["after"])
    record["last_decode_after_retention"] = dict(
        processed_tokens=last["processed_tokens"], request_owned_blocks=owned,
        request_owned_slot_capacity=owned * manager.block_size,
        request_owned_page_capacity_bytes=owned * spec.page_size_bytes * len(group.layer_names),
    )


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model-dir", type=Path, required=True)
    parser.add_argument("--condition", choices=("full", "sliding"), required=True)
    parser.add_argument("--device", choices=("cpu", "cuda"), default="cuda")
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    record = dict(
        status="FAILED", condition=args.condition, platform=platform.platform(),
        requested_vllm_version=VERSION, requested_device=args.device, prompt=PROMPT, trace=[],
    )
    with args.output.open("x", encoding="utf-8", newline="\n") as stream:
        try:
            run(args, record)
            record["status"] = "PASS"
        except Exception as exc:
            record["error"] = f"{type(exc).__name__}: {exc}"
            record["traceback"] = traceback.format_exc()
        json.dump(record, stream, indent=2)
        stream.write("\n")
    print(f"{record['status']} {args.output}", file=sys.stderr)
    return 0 if record["status"] == "PASS" else 1


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
from importlib.metadata import PackageNotFoundError, version
def installed_vllm():
    try:
        return version("vllm")
    except PackageNotFoundError:
        return None

if installed_vllm() == "0.26.0":
    python = sys.executable
else:
    env = workspace / "venv"
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(env)], check=True)
    python = str(env / "bin/python")
    subprocess.run([sys.executable, "-m", "pip", "--python", python, "install", "--no-cache-dir", "--disable-pip-version-check",
                    "vllm==0.26.0"], check=True)
with (out / "packages.txt").open("w") as log:
    subprocess.run([sys.executable, "-m", "pip", "--python", python, "freeze"], stdout=log, check=True)
fetch = """from huggingface_hub import snapshot_download
import sys
snapshot_download('Qwen/Qwen2.5-0.5B-Instruct', revision='7ae557604adf67be50417f59c2c2f167def9a775', local_dir=sys.argv[1], allow_patterns=['*.json','*.safetensors','*.txt'])"""
model_dir = workspace / "clean-qwen"
subprocess.run([python, "-c", fetch, str(model_dir)], check=True)

In [ ]:
arms = {}
for condition in ("full", "sliding"):
    output = out / (condition + ".json")
    with (out / (condition + ".log")).open("w", buffering=1) as log:
        attempt = subprocess.run([python, "-m", "scripts.run_vllm_sliding_window_probe",
                                  "--condition", condition, "--device", "cuda",
                                  "--model-dir", str(model_dir), "--output", str(output)],
                                 stdout=log, stderr=subprocess.STDOUT)
    try:
        arm = json.loads(output.read_text())
    except (OSError, ValueError):
        arm = {"status": "FAILED", "error": "Process exited before completing the trace"}
    arms[condition] = arm
    print(condition, attempt.returncode, arm.get("status"), arm.get("error", ""), flush=True)

full, sliding = arms["full"], arms["sliding"]
checks = {"both_traces_pass": all(a.get("status") == "PASS" for a in arms.values())}
if checks["both_traces_pass"]:
    for key in ("hardware","model_revision","verified_snapshot_hashes","model_dtype","vllm_version",
                "attention_backend","prompt_token_ids","seed","max_output_tokens"):
        checks[key] = full[key] == sliding[key]
    checks["first_token"] = full["generated_token_ids"][0] == sliding["generated_token_ids"][0]
    checks["live_eviction_observed"] = full["verified_release_steps"] == 0 and sliding["verified_release_steps"] > 0
summary = {"status": "PASS" if all(checks.values()) else "FAILED", "checks": checks,
           "release_steps": {k:a.get("verified_release_steps") for k,a in arms.items()}}
(out / "paired_summary.json").write_text(json.dumps(summary, indent=2))
print(summary)
# Preserve outputs for diagnosis even if one condition fails the scientific check.
